# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```

**Note:** All record sets, fields, and other entities are referenced by their Croissant `@id`.

In [ ]:
# Ensure mlcroissant library is available
!pip install mlcroissant

## 1. Data Loading
Load the Croissant metadata and records for the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Author IDs:", [author['@id'] for author in metadata.author])
print("RecordSet IDs:", getattr(metadata, 'recordSet', []))

## 2. Data Overview
Explore available record sets and fields (by their `@id`) defined in the dataset schema.

Note: For this dataset, record sets are defined in the schema and typically describe tables of tabular data. We'll list them, then review their structure.

In [ ]:
# Extract all RecordSet @id from the Croissant schema
from pprint import pprint
recordset_ids = []
croissant_jsonld = dataset.croissant()
for item in croissant_jsonld:
    if item.get('@type') == 'RecordSet':
        recordset_ids.append(item['@id'])

print("Available Record Sets by @id:")
for idx, rsid in enumerate(recordset_ids):
    print(f"{idx+1}. {rsid}")

# For each record set, list the fields and their @ids
for rsid in recordset_ids:
    print(f"\nRecordSet @id: {rsid}")
    for item in croissant_jsonld:
        if item.get('@id') == rsid:
            fields = item.get('field', [])
            if not isinstance(fields, list):
                fields = [fields]
            print(' Fields:')
            for f in fields:
                if isinstance(f, dict):
                    print(f"   - {f.get('@id')}: {f.get('name', 'No name')}")
                elif isinstance(f, str):
                    print(f"   - {f}")

## 3. Data Extraction
Load records from each record set into DataFrames for further analysis. Each record set and each field is referenced by its `@id`.

In [ ]:
# Prepare list of record set @ids (update if record sets change)
record_sets = recordset_ids  # From previous step
dataframes = {}

for rsid in record_sets:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"RecordSet {rsid} loaded with shape {df.shape}")

# Display columns for one record set
if record_sets:
    record_set_to_show = record_sets[0]
    print(f"Columns for {record_set_to_show}: {dataframes[record_set_to_show].columns.tolist()}")
    display(dataframes[record_set_to_show].head())

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps, such as filtering records, normalizing numeric fields, and grouping by categorical variables.

We'll select a numeric field (referenced by its `@id`) and a group field from one record set.

In [ ]:
# Example: Assume Table 2 has hormone level columns, Table 1 has clinical features
# Replace these ids with the actual @ids found above!

# Use the first record set for demonstration
if record_sets:
    rsid = record_sets[0]
    df = dataframes[rsid]
    print(f"Operating on RecordSet {rsid}")

    # Identify numeric fields (@id) in this record set
    numeric_candidate = None
    for col in df.columns:
        # Try to guess numeric column by simple dtype test
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_candidate = col
            break

    if numeric_candidate:
        print(f"Using numeric field @id: {numeric_candidate}")

        # Filtering by threshold
        threshold = df[numeric_candidate].mean() if df.shape[0] else 0
        filtered_df = df[df[numeric_candidate] > threshold]
        print(f"Filtered records with {numeric_candidate} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_candidate}_normalized"] = (
            filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()
        ) / (filtered_df[numeric_candidate].std() + 1e-10)
        print(f"Normalized {numeric_candidate} for filtered records:")
        display(filtered_df[[numeric_candidate, f"{numeric_candidate}_normalized"]].head())

        # Grouping by a categorical field (if available)
        group_candidate = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_candidate:
                group_candidate = col
                break
        if group_candidate:
            grouped_df = filtered_df.groupby(group_candidate)[numeric_candidate].mean().reset_index()
            print(f"Grouped data by {group_candidate} (@id):")
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions. We'll plot the histogram for the selected numeric field and, if possible, a boxplot grouped by the group field (by `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_sets:
    rsid = record_sets[0]
    df = dataframes[rsid]
    # Choose numeric and categorical field as before
    numeric_candidate = None
    group_candidate = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_candidate = col
            break
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_candidate:
            group_candidate = col
            break
    if numeric_candidate:
        plt.figure(figsize=(7, 4))
        sns.histplot(df[numeric_candidate], kde=True)
        plt.title(f"Distribution of numeric field (@id): {numeric_candidate}")
        plt.xlabel(numeric_candidate)
        plt.show()
    if numeric_candidate and group_candidate:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_candidate], y=df[numeric_candidate])
        plt.title(f"Boxplot of {numeric_candidate} by {group_candidate} (@id)")
        plt.xlabel(group_candidate)
        plt.ylabel(numeric_candidate)
        plt.show()

## 6. Conclusion
In this notebook, we successfully loaded and explored the FAIR^2 clinical dataset using the `mlcroissant` library, referencing each record set, field, and column by its Croissant `@id`. We examined metadata, performed basic EDA, and visualized distributions of clinical/laboratory features. This dataset provides structured insight into rare NC-21OHD-related infertility, but is limited in scope and sample size. The Croissant format ensures FAIR principles and reproducible data access for academic analyses.

For further exploration, consult the Croissant schema to identify more fields or record sets, and experiment with advanced filtering and modeling as permitted by the dataset structure and size.